In [1]:
pip install numpy

  Using cached numpy-2.5.4-cp312-cp312-win_amd64.whl.metadata (6.6 kB)
Using cached numpy-2.5.4-cp312-cp312-win_amd64.whl (12.6 MB)
Note: you may need to restart the kernel to use updated packages.


In [1]:
pip install pandas

   ---------------------------------------- 0.0/10.7 MB ? eta -:--:--
    --------------------------------------- 0.3/10.7 MB ? eta -:--:--
   -------- ------------------------------- 2.4/10.7 MB 9.0 MB/s eta 0:00:01
   ----------------------- ---------------- 6.3/10.7 MB 13.3 MB/s eta 0:00:01
   ------------------------------------- -- 10.0/10.7 MB 14.8 MB/s eta 0:00:01
   ---------------------------------------- 10.7/10.7 MB 14.0 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [1]:
pip install matplotlib

  Using cached matplotlib-3.11.2-cp312-cp312-win_amd64.whl.metadata (80 kB)
  Using cached contourpy-1.4.0-cp312-cp312-win_amd64.whl.metadata (3.8 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.67.0-cp312-cp312-win_amd64.whl.metadata (139 kB)
  Using cached kiwisolver-1.5.1-cp312-cp312-win_amd64.whl.metadata (5.2 kB)
  Using cached pyparsing-3.3.3-py3-none-any.whl.metadata (5.9 kB)
Using cached matplotlib-3.11.2-cp312-cp312-win_amd64.whl (9.3 MB)
Using cached contourpy-1.4.0-cp312-cp312-win_amd64.whl (234 kB)
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
Using cached fonttools-4.67.0-cp312-cp312-win_amd64.whl (2.5 MB)
Using cached kiwisolver-1.5.1-cp312-cp312-win_amd64.whl (70 kB)
   ---------------------------------------- 0.0/7.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/7.3 MB ? eta -:--:--
   ---------- ----------------------------- 1.8/7.3 MB 7.7 MB/s eta 0:00:01
   ----------------------- --------------

# Shared data contract (TV1/TV2/TV3)

This document defines the shared input/output interface for data generation,
training, validation, and experiments. Paths below are relative to the project
root; code must not depend on an individual developer's absolute path.

## Processed files and schema

`python -m src.data.generate_data` creates these files under
`data/processed/`:

| File | Default rows | Usage |
| --- | ---: | --- |
| `train_pool.csv` | 2,000 | Training pool; experiment A uses prefixes of this same file. |
| `val.csv` | 400 | Training monitoring and configuration/model selection. |
| `test.csv` | 100 | Final evaluation only; never use to select a model or configuration. |
| `metadata.json` | — | Split seeds, sizes, ranges, Gaussian mean/covariance, and column names. |

Each CSV has exactly three columns in this order, with no saved row index:

| Column | Meaning | Loader dtype/shape |
| --- | --- | --- |
| `x1` | First input coordinate, sampled uniformly from `[-2, 2]`. | `X[:, 0]`, float32 |
| `x2` | Second input coordinate, sampled uniformly from `[-4, 4]`. | `X[:, 1]`, float32 |
| `y` | Gaussian 2D probability density at `(x1, x2)`. | `y`, float32 |

The input and target arrays returned by the loader have shapes `X: (N, 2)` and
`y: (N, 1)`. The CSV's textual/numeric representation is not a model dtype
guarantee; consumers should use the shared loader, which explicitly converts
both arrays to finite `numpy.float32`.

## Canonical loader interface

```python
from src.data.load_data import load_data

X_train, y_train = load_data("train_pool", n_samples=400)
X_val, y_val = load_data("val")
X_test, y_test = load_data("test")
```

Signature:

```python
load_data(
    split: Literal["train_pool", "val", "test"],
    n_samples: int | None = None,
    *,
    data_dir: str | Path | None = None,
) -> tuple[np.ndarray, np.ndarray]
```

- When omitted, `data_dir` resolves to this project's `data/processed/`
  directory based on the loader file location, not the caller's working
  directory.
- `n_samples` is valid only for `train_pool`. It returns the first N rows, not
  a newly sampled subset. This keeps experiment A's 200, 400, ..., 2,000 sample
  sets nested and reproducible.
- Unknown split names, malformed CSV schema, empty files, non-finite values,
  or invalid sample counts raise an error.
- Convert arrays to Torch tensors when constructing TV2's
  `TensorDataset`/`DataLoader`, e.g.
  `TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train))`.
  `train_one_run` consumes batches of `(X, y)` tensors.

This interface and schema are the repository's canonical TV1-to-TV2/TV3
contract. Experiment runners should read the shared files through
`src.data.load_data`, rather than duplicating paths or CSV parsing.

## Reproducibility and evaluation rules

- Default data generation uses train/validation/test seeds `42`, `43`, and
  `44`, respectively. The generated values and sizes are recorded in
  `metadata.json`.
- The default split sizes match `configs/baseline.json`: 2,000 / 400 / 100.
- Experiment A uses fixed prefixes from the train pool. Do not shuffle or
  independently resample for each sample-size condition unless the experiment
  protocol is explicitly changed.
- Use validation for early stopping and configuration/model selection. Keep
  `test.csv` held out until final evaluation.
- `experiment_seeds` in the baseline configuration are experiment/run seeds;
  they do not change the already generated data split. Split seeds belong to
  the metadata.

See [the data module README](../src/data/README.md) for generation and loading
examples.


# src/data — TV1

## Nhiệm vụ
- `gaussian.py`: cài đặt hàm mật độ Gaussian 2 chiều theo công thức; kiểm tra đầu vào và tham số cần thiết.
- `generate_data.py`: lấy mẫu x1, x2 trong miền đề bài, tính y, chia train/validation/test độc lập và lưu CSV.
- CSV thống nhất các cột `x1,x2,y`; metadata lưu seed, miền, μ, C và kích thước split.

## Dữ liệu 
- Các tệp được lưu trong data/processed/
    - `train_pool.csv` (2.000 mẫu): Tệp huấn luyện, thực nghiệm A dùng các prefix cố định của tập này 
    - `val.csv` (500 mẫu): Chọn cấu hình và theo dõi quá trình huấn luyện 
    - `test.csv` (100 mẫu): Đánh giá cuối cùng, không dùng để chọn cấu hình 
    - `metadata.json`: Lưu seed, miền lấy mẫu, tham số Gaussian và kích thước các tập 

## Cách tải dữ liệu

```python
from src.data.load_data import load_data

X_train, y_train = load_data("train_pool", n_samples=400)
X_val, y_val = load_data("val")
X_test, y_test = load_data("test")
```

- Kiểm tra dữ liệu

```python
print(X_train.shape, y_train.shape)  # (400, 2), (400, 1)
print(X_val.shape, y_val.shape)      # (500, 2), (500, 1)
print(X_test.shape, y_test.shape)    # (100, 2), (100, 1)
```

# Bản nháp phần đóng góp — TV1

## 1. Phạm vi đóng góp

TV1 phụ trách xây dựng hàm Gaussian 2D, sinh và chia dữ liệu phục vụ thực nghiệm, khảo sát dữ liệu ban đầu (EDA) và xây dựng data contract nhằm thống nhất giao diện dữ liệu giữa các thành viên trong nhóm.

## 2. Phương pháp và code thực tế

Hàm Gaussian 2D được triển khai trong `src/data/gaussian.py` để tính giá trị mục tiêu \(y\) từ hai biến đầu vào \(x_1\) và \(x_2\). Dữ liệu đầu vào được lấy mẫu theo phân phối đều trên hai miền \(x_1 \in [-2,2]\) và \(x_2 \in [-4,4]\).

Quy trình sinh dữ liệu được triển khai trong `src/data/generate_data.py`. Theo cấu hình mặc định, chương trình tạo ba tập dữ liệu độc lập gồm `train_pool.csv` với 2.000 mẫu, `val.csv` với 400 mẫu và `test.csv` với 100 mẫu. Các tập sử dụng seed lần lượt là 42, 43 và 44 nhằm hỗ trợ tái lập dữ liệu. File `metadata.json` lưu lại seed, miền lấy mẫu, tham số Gaussian, kích thước các tập và tên các cột dữ liệu.

EDA được thực hiện nhằm khảo sát phân bố của hai biến đầu vào, giá trị mục tiêu và sự khác biệt giữa các tập dữ liệu. Data contract quy định schema CSV gồm ba cột `x1,x2,y`, không có cột index thừa. Sau khi đọc dữ liệu, `features` có kiểu `float32`, shape `(N,2)` và `targets` có kiểu `float32`, shape `(N,1)`.

## 3. Kết quả và log minh chứng

**Kết quả theo cấu hình mặc định trong mã nguồn:**

- Quy mô dữ liệu dự kiến: 2.500 mẫu, gồm 2.000 mẫu train pool, 400 mẫu validation và 100 mẫu test.
- Schema dữ liệu: `x1,x2,y`.
- Seed mặc định: 42 cho train pool, 43 cho validation và 44 cho test.
- File đầu ra dự kiến: `train_pool.csv`, `val.csv`, `test.csv` và `metadata.json` trong `data/processed/`.

Các nội dung trên được suy ra từ cấu hình mã nguồn, chưa thay thế cho log xác nhận chạy thành công. Cần bổ sung log thực tế sau khi chạy:

`python -m src.data.generate_data`

Minh chứng cần lưu gồm thông báo thực thi, kiểm tra sự tồn tại của bốn file đầu ra, số dòng thực tế của từng CSV, tên cột và kết quả kiểm tra kiểu dữ liệu sau khi đọc bằng `load_data()`.

## 4. Hình và bảng cần chèn

- `data_splits_3d.png`: biểu diễn phân bố các điểm `(x1,x2,y)` trên ba tập dữ liệu.
- `data_true_surface.png`: biểu diễn bề mặt hàm Gaussian 2D theo công thức và tham số đã thống nhất.
- `data_distributions.png`: so sánh phân phối của `x1`, `x2` và `y` giữa các tập.
- Bảng thống kê dữ liệu: trình bày số mẫu, số cột, schema và seed của từng tập.
- Bảng data contract: tổng hợp quy ước kiểu dữ liệu, shape và interface đọc dữ liệu dùng chung giữa các thành viên.

Chỉ chèn các hình và bảng sau khi đã tạo, kiểm tra và xác nhận chúng phản ánh đúng dữ liệu thực tế.

## 5. Giới hạn và ghi chú

Dữ liệu được sinh nhân tạo theo công thức Gaussian 2D nên chưa phản ánh đầy đủ đặc điểm của dữ liệu thực tế. Kết quả thực nghiệm phụ thuộc vào công thức Gaussian, miền lấy mẫu, seed và cấu hình sinh dữ liệu.

Việc sử dụng các seed riêng giúp tái lập quy trình sinh dữ liệu nhưng không tự động bảo đảm mọi đặc tính thống kê giữa các tập đều giống nhau. Các thực nghiệm với kích thước train khác nhau cần lấy prefix từ cùng `train_pool.csv` để bảo đảm tính nhất quán. Tập test gồm 100 mẫu chỉ được sử dụng cho đánh giá cuối cùng, không dùng để lựa chọn cấu hình mô hình.

Cần kiểm tra sự thống nhất giữa `data_contract.md`, `metadata.json` và interface `load_data()` trong mã nguồn trước khi tích hợp với các thành viên phụ trách MLP và huấn luyện.

## 6. Tài liệu tham khảo đã sử dụng

- Mã nguồn nội bộ `src/data/gaussian.py`: định nghĩa hàm Gaussian 2D.
- Mã nguồn nội bộ `src/data/generate_data.py`: sinh, chia và lưu dữ liệu.
- `docs/data_contract.md`: quy ước schema và interface dữ liệu của nhóm.
- `data/processed/metadata.json`: thông tin cấu hình và siêu dữ liệu của bộ dữ liệu được sinh.

Đây là các artefact nội bộ của dự án, không phải tài liệu tham khảo học thuật bên ngoài. Nếu báo cáo yêu cầu tài liệu tham khảo học thuật, cần bổ sung nguồn thực tế đã tham khảo để xây dựng công thức Gaussian 2D và phương pháp sinh dữ liệu.


# Bản nháp phần đóng góp — TV1

## Phạm vi
Gaussian, sinh/chia dữ liệu, EDA, data contract.
TV1 phụ trách: 
- Xây dựng hàm Gaussian 2D
- Sinh và chia dữ liệu phục vụ thực nghiệm
- Khảo sát dữ liệu ban đầu (EDA) 
- Cập nhật data contract nhằm thống nhất giao diện dữ liệu giữa các thành viên trong nhóm.

## Nội dung cần điền
### Phương pháp/code thực tế
**Xây dựng hàm Gaussian 2D (`src/data/gaussian.py`):** tính giá trị mục tiêu \(y\) từ hai biến đầu vào \(x_1\) và \(x_2\). Dữ liệu đầu vào được lấy mẫu theo phân phối đều trên hai miền \(x_1 \in [-2,2]\) và \(x_2 \in [-4,4]\).

**Quy trình sinh dữ liệu (`src/data/generate_data.py`):** Theo cấu hình mặc định, chương trình tạo ba tập dữ liệu độc lập gồm:
    - `train_pool.csv`: 2.000 mẫu, seed: 42 
    `val.csv`: 400 mẫu, seed: 43
    `test.csv`: 100 mẫu, seed: 44
File `metadata.json` lưu lại seed, miền lấy mẫu, tham số Gaussian, kích thước các tập và tên các cột dữ liệu.

**EDA:** khảo sát phân bố của hai biến đầu vào, giá trị mục tiêu và sự khác biệt giữa các tập dữ liệu. 

**Data contract.md:** quy định schema CSV gồm ba cột `x1,x2,y`, không có cột index thừa. Sau khi đọc dữ liệu, `features` có kiểu `float32`, shape `(N,2)` và `targets` có kiểu `float32`, shape `(N,1)`.

### Kết quả/log minh chứng

- Hình/bảng cần chèn:
- Giới hạn/ghi chú:
- Tài liệu tham khảo đã sử dụng:

Không đưa số liệu giả; thay placeholder bằng kết quả chạy thực tế.


# TV1 — Dữ liệu và EDA

## Phạm vi
Gaussian, sinh/chia dữ liệu, EDA, data contract.
TV1 phụ trách: 
- Xây dựng hàm Gaussian 2D
- Sinh và chia dữ liệu phục vụ thực nghiệm
- Khảo sát dữ liệu ban đầu (EDA) 
- Cập nhật data contract nhằm thống nhất giao diện dữ liệu giữa các thành viên trong nhóm.

## Nội dung cần điền
### 1. Phương pháp/code thực tế
- Triển khai hàm Gaussian 2D trong `src/data/gaussian.py`.
- Sinh và chia dữ liệu trong `src/data/generate_data.py`, gồm train pool (2.000 mẫu), validation (400 mẫu), test (100 mẫu); seed lần lượt 42, 43, 44.
- Thống nhất schema CSV `x1,x2,y`, kiểu dữ liệu sau khi đọc và interface giữa các thành viên. 
- Thực hiện EDA để khảo sát phân phối dữ liệu và trực quan hóa hàm Gaussian.

### 2. Kết quả/log minh chứng
Theo cấu hình mặc định, tổng dữ liệu dự kiến là 2.500 mẫu. Cần chạy `python -m src.data.generate_data` và kiểm tra các file CSV, `metadata.json` để xác nhận kết quả.

### 3. Hình/bảng cần chèn
- `data_splits_3d.png`: phân bố dữ liệu theo ba tập.
- `data_true_surface.png`: bề mặt Gaussian 2D.
- `data_distributions.png`: phân phối `x1`, `x2`, `y`.
- Bảng thống kê kích thước, seed và schema các tập dữ liệu.

### 4. Giới hạn/ghi chú
Dữ liệu được sinh nhân tạo, chưa phản ánh đầy đủ dữ liệu thực tế. Các thực nghiệm dùng prefix của cùng train pool, validation dùng chọn cấu hình, test 100 mẫu chỉ dùng đánh giá cuối.

### 5. Tài liệu đã sử dụng
- `src/data/gaussian.py`.
- `src/data/generate_data.py`.
- `docs/data_contract.md`.
- `data/processed/metadata.json`.



